# Indigenous lands and resource rights 

The data processed here comes from Landmark. The processing includes reducing the number of fields, a geometry simplification and the conversion to mbtiles. The resulting mbtiles files are uploaded manually in Mapbox project.

### Set up
#### Library import

In [ ]:
import os
import geopandas as gpd
from pathlib import Path
import logging
import subprocess

# Create a logger
logger = logging.getLogger(__name__)

# Set the log level to INFO
logger.setLevel(logging.INFO)

#### Utils

In [ ]:
# Function to create MBTILES from a GeoJSON file
def create_mbtiles(
    source_path: Path,
    output_path: Path,
    layer_name: str,
    max_zoom: int,
    opts="--read-parallel --no-tile-compression -s EPSG:4326 -B4",
):
    """
    Use tippecanoe to create pbf tiles at dest_path from source_path (geojson).
    layer_name is used for the name of the layer in the MBTILE.
    Regex file path (/*.geojson) is supported for source_path.
    This function replaces the previous two functions (create_mbtiles & mbtile_to_pbf).

    More info: https://github.com/mapbox/tippecanoe#options

    Args:
        source_path (Path): path to source geojson
        output_path (Path): path to output .mbtiles
        layer_name (str): name of layer in the MBTILE
        max_zoom (int): max zoom level
        opts (str): options for tippecanoe

    Returns:
        (int): 0 if the file was created successfully, 1 if the file creation failed.
    """
    try:
        opts += f" -z{max_zoom}"
        cmd = f"tippecanoe -o {output_path} -l {layer_name} {opts} {source_path}"
        logger.info(f"Processing: {cmd}")
        r = subprocess.call(cmd, shell=True)
        if r == 0:
            logger.info("Task created")
        return r

    except Exception as e:
        logger.error(e)
        return 1

### Processing
#### Load and prepare data

In [ ]:
# Load and explore the indigenous land data
lands = gpd.read_file("../data/raw/Indigenous_comm_lands_v202411/CommunityLevelData_poly_202411.shp")
rights = gpd.read_file("../data/raw/Resource_rights_v202411/CommunityResources_202411.shp")

In [ ]:
# Keep only relevant fields
relevant_cols = ['Name', 'Identity', 'Form_Rec', 'Doc_Status', 'Country', 'ISO_Code', 'Category', 'Data_Ctrb', 'Data_Src', 'geometry']

lands2 = lands[relevant_cols]
rights2 = rights[relevant_cols]

In [ ]:
# Change the crs to EPSG:4326
lands2 = lands2.to_crs(epsg=4326)
rights2 = rights2.to_crs(epsg=4326)

In [ ]:
# Divide lands2 layer in two layers according to Identity field
indigenous = lands2[lands2['Identity'] == 'Indigenous']
community = lands2[lands2['Identity'] == 'Community']

In [ ]:
# Save the data
indigenous.to_file("../data/processed/indigenous/indigenous_lands_v202411.geojson", driver='GeoJSON')
community.to_file("../data/processed/indigenous/community_lands_v202411.geojson", driver='GeoJSON')
rights2.to_file("../data/processed/indigenous/CommunityResources_202411.geojson", driver='GeoJSON')

#### Simplify geometries

In [ ]:
!mapshaper-xl 16gb -i /Users/sofia/Documents/Repos/sci_team_data_bank/Projects/global_rangelands_data_platform/data/processed/indigenous/indigenous_lands_v202411.geojson snap \
    -simplify 50% weighting=0.3 planar keep-shapes \
    -filter-islands min-vertices=3 remove-empty \
    -filter-slivers remove-empty \
    -clean rewind \
    -o /Users/sofia/Documents/Repos/sci_team_data_bank/Projects/global_rangelands_data_platform/data/processed/indigenous/indigenous_lands_v202411_simplified.geojson format=geojson 

!mapshaper-xl 16gb -i /Users/sofia/Documents/Repos/sci_team_data_bank/Projects/global_rangelands_data_platform/data/processed/indigenous/community_lands_v202411.geojson snap \
    -simplify 50% weighting=0.3 planar keep-shapes \
    -filter-islands min-vertices=3 remove-empty \
    -filter-slivers remove-empty \
    -clean rewind \
    -o /Users/sofia/Documents/Repos/sci_team_data_bank/Projects/global_rangelands_data_platform/data/processed/indigenous/community_lands_v202411_simplified.geojson format=geojson 


!mapshaper-xl 16gb -i /Users/sofia/Documents/Repos/sci_team_data_bank/Projects/global_rangelands_data_platform/data/processed/indigenous/CommunityResources_202411.geojson snap \
    -simplify 70% weighting=0.3 planar keep-shapes \
    -filter-islands min-vertices=3 remove-empty \
    -filter-slivers remove-empty \
    -clean rewind \
    -o /Users/sofia/Documents/Repos/sci_team_data_bank/Projects/global_rangelands_data_platform/data/processed/indigenous/CommunityResources_202411_simplified.geojson format=geojson 

#### Create mbtiles

In [ ]:
path = '../data/processed/indigenous'

create_mbtiles(
    os.path.join(path, "indigenous_lands_v202411_simplified.geojson"),
    os.path.join(path, "Indigenous_lands.mbtiles"),
    "indigenous_lands",
    12,
    "--force --read-parallel -zg -Z2 --drop-densest-as-needed --extend-zooms-if-still-dropping",
)

create_mbtiles(
    os.path.join(path, "community_lands_v202411_simplified.geojson"),
    os.path.join(path, "Community_lands.mbtiles"),
    "community_lands",
    12,
    "--force --read-parallel -zg -Z2 --drop-densest-as-needed --extend-zooms-if-still-dropping",
)

create_mbtiles(
    os.path.join(path, "CommunityResources_202411_simplified.geojson"),
    os.path.join(path, "Resources_rights.mbtiles"),
    "resources_rights",
    12,
    "--force --read-parallel -zg -Z2 --drop-densest-as-needed --extend-zooms-if-still-dropping",
)

In [ ]:
lands2['Form_Rec'].unique()